# 🧭 Protocol splitters: cross-validation and evaluation harnesses

Welcome! Unlike most `chemsplit` families, the `protocol` splitters don't invent a new way to decide *which* records go where — they don't cluster, group by scaffold, or cut by date. Instead, each one **wraps or composes another splitter** to implement a standard ML evaluation scheme on top of it: k-fold cross-validation, nested CV, repeated resampling, three-way train/valid/test carving, an applicability-domain distance curve, or forcing an external dataset to be the test set. Reach for this family when the question isn't "what's a good split criterion?" (that's every other family) but "how do I turn a split criterion into a rigorous evaluation protocol?"

This notebook builds one shared pool of 40 synthetic SMILES, then walks through all 6 classes in turn, each with its own worked example and a short summary of the `SplitResult`(s) it returns.

<a id="0"></a>
### 🛠️ Setup

In [1]:
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")  # silence RDKit's standardizer log spam during split_result()

from chemsplit.registry import get_splitter
from chemsplit.splitters.protocol import (
    ApplicabilityDomainSplitter,
    ExternalHoldoutSplitter,
    GroupKFoldSplitter,
    NestedCVSplitter,
    RepeatedSplitter,
    ThreeWaySplitter,
)

SMILES_POOL = [
    "CCO", "CCN", "CCC", "CCCl", "CCBr", "c1ccccc1", "c1ccncc1", "c1ccccc1O",
    "c1ccccc1N", "CC(=O)O", "CC(=O)N", "CCCCO", "CCCCN", "CCCCC", "c1ccoc1",
    "c1ccsc1", "C1CCCCC1", "C1CCNCC1", "C1CCOCC1", "CC(C)O", "CC(C)N",
    "CCCCCCO", "CCCCCCN", "c1ccc2ccccc2c1", "CC(=O)OC", "CCOC(=O)C",
    "CCCCCCCCO", "CCCCCCCCN", "c1ccc(F)cc1", "c1ccc(Cl)cc1", "c1ccc(Br)cc1",
    "CN1CCCCC1", "O=C1CCCCC1", "OC1CCCCC1", "NC1CCCCC1", "CC1CCCCC1",
    "c1cc2ccccc2cc1", "CCN(CC)CC", "CCOCC", "CC#N", "CC=O",
]

def summarize(results, label):
    print(f"{label}: {len(results)} SplitResult(s)")
    for i, r in enumerate(results):
        extra = {k: v for k, v in r.metadata.items() if k in ("repeat_index", "band_index", "distance_range")}
        print(f"  [{i}] train={r.train.size} valid={r.valid.size} test={r.test.size} discard={r.discard.size} {extra}")

<a id="1"></a>
## 1. 🧩 GroupKFoldSplitter — group-aware cross-validation

K-fold cross-validation whose folds respect a grouping (e.g. Murcko scaffold), so no group straddles a fold boundary — the group-aware analogue of plain k-fold. It turns any grouping into a full cross-validation protocol, testing every record exactly once under group-level holdout.

| Parameter | Meaning |
|---|---|
| `grouper` | Supplies group labels (here, a registry id resolved to a grouping splitter, e.g. `"murcko_scaffold"`) |
| `n_splits` | Number of folds, or `"auto"` for one fold per group |
| `random_state` | Seeds the greedy bin-packing fold assignment |

> 💡 **Advantages:**
> - Turns any grouping — scaffold, cluster, assay, series, party — into a full cross-validation protocol, testing every record exactly once under group-level holdout.
> - Greedy largest-first bin packing keeps folds much closer to equal size than random group assignment, which matters when one group holds a large share of the data.
> - Accepts externally computed group labels, so a grouping from any other tool composes cleanly.
>
> ⚠️ **Pitfalls:**
> - Fold sizes stay unequal whenever group sizes are, so per-fold metrics come from different sample sizes — report fold sizes alongside scores.
> - The grouping determines everything — `GroupKFold` over Murcko scaffolds inherits every weakness of `murcko_scaffold`.
> - Greedy bin packing is deterministic but not optimal, so an adversarial group-size distribution can still yield a badly skewed fold.
> - With few groups, `n_splits` is capped and folds end up large and highly correlated.

In [2]:
s = GroupKFoldSplitter(grouper="murcko_scaffold", n_splits=3, random_state=0)
results = s.split_result(SMILES_POOL)
summarize(results, "GroupKFoldSplitter")

/mnt/c/Users/Olivier/Documents/GitHub/chemsplit/src/chemsplit/base.py:632: DuplicateWarning: 1 duplicate key(s) affecting 2 record(s)
  pipeline_result = preprocess.run_pipeline(


GroupKFoldSplitter: 3 SplitResult(s)
  [0] train=27 valid=0 test=14 discard=0 {}
  [1] train=27 valid=0 test=14 discard=0 {}
  [2] train=28 valid=0 test=13 discard=0 {}


Three folds, each holding out a disjoint set of scaffold groups as its test partition — no scaffold appears in both a fold's train and test.

<a id="2"></a>
## 2. 🎯 ThreeWaySplitter — a genuine train/valid/test split

Applies a wrapped splitter's criterion at **both** boundaries (train↔valid and (train∪valid)↔test), producing a proper three-way split from a splitter that natively only knows train/test. Internally: stage 1 carves `(train+valid)` from `test`; stage 2 re-applies a freshly-seeded clone of the same splitter to the `(train+valid)` subset alone, to carve `train` from `valid`.

| Parameter | Meaning |
|---|---|
| `base_splitter` | The splitter whose criterion carves each boundary — here, plain `"random"` |
| `train_size` / `valid_size` / `test_size` | Target fractions for the three partitions |

> 💡 **Advantages:**
> - Closes the leak where a scaffold-split test set sits behind a randomly split validation set — an easy validation set otherwise selects a model optimised for interpolation.
> - Applying the same criterion to both boundaries makes the validation score an honest, if optimistic, preview of the test score.
> - Exact index remapping lets the wrapper compose with every splitter without special cases.
>
> ⚠️ **Pitfalls:**
> - A structurally split validation set is smaller and harder, so early stopping triggers sooner and hyperparameter choices get noisier.
> - Applying a group-forming criterion twice compounds the size drift.

In [3]:
s = ThreeWaySplitter(base_splitter="random", train_size=0.5, valid_size=0.25, test_size=0.25, random_state=0)
results = s.split_result(SMILES_POOL)
summarize(results, "ThreeWaySplitter")

ThreeWaySplitter: 1 SplitResult(s)
  [0] train=21 valid=10 test=10 discard=0 {}


<a id="3"></a>
## 3. 🔁 RepeatedSplitter — estimating split-induced variance

Repeats a wrapped splitter under multiple independently-derived seeds, to estimate how much of a downstream model's performance spread is just an artefact of *which* split you happened to draw — frequently more than the difference between the models being compared.

| Parameter | Meaning |
|---|---|
| `base_splitter` | The splitter to repeat — here, plain `"random"` |
| `n_repeats` | Number of independently-seeded repeats |

> 💡 **Advantages:**
> - Between-split variance frequently exceeds between-model differences on chemical data, so without repeats a model comparison isn't real evidence.
> - Detects and warns when repetition is meaningless because the wrapped splitter is deterministic.
>
> ⚠️ **Pitfalls:**
> - Repeating a *group-forming* splitter with `group_assignment="greedy_desc"` yields identical results each time — only `"random"` assignment or a seeded clusterer actually varies.
> - The spread across repeats measures split variance, not model uncertainty — don't report it as a model confidence interval.
> - Cost multiplies directly, so `O(n²)` splitters get expensive fast.

In [4]:
s = RepeatedSplitter(base_splitter="random", n_repeats=4, random_state=0)
results = s.split_result(SMILES_POOL)
summarize(results, "RepeatedSplitter")
print("get_n_splits() ->", s.get_n_splits())

RepeatedSplitter: 4 SplitResult(s)
  [0] train=33 valid=0 test=8 discard=0 {'repeat_index': 0}
  [1] train=33 valid=0 test=8 discard=0 {'repeat_index': 1}
  [2] train=33 valid=0 test=8 discard=0 {'repeat_index': 2}
  [3] train=33 valid=0 test=8 discard=0 {'repeat_index': 3}
get_n_splits() -> 4


<a id="4"></a>
## 4. 🪆 NestedCVSplitter — hyperparameter tuning without leakage

For each outer fold's training set, an inner splitter further carves out a validation slice — the sanctioned way to combine hyperparameter selection with an honest outer test score. Outer test folds are never seen by any tuning decision, which is the whole point.

| Parameter | Meaning |
|---|---|
| `outer_splitter` | Determines the outer train/test folds — here, a 3-fold `k_fold` |
| `inner_splitter` | Run on each outer fold's train subset only, to produce inner train/valid pairs |

> 💡 **Advantages:**
> - The only honest protocol when hyperparameters are tuned at all — outer test folds are never seen by any tuning decision.
> - Composes any outer criterion with any inner criterion, so the tuning distribution can match the evaluation distribution.
>
> ⚠️ **Pitfalls:**
> - Expensive: `outer × (1 + inner)` splits and their model fits.
> - Skipping it and reporting the best inner-loop score is the most common silent source of optimism in QSAR papers.
> - Estimates the performance of the *whole tuning procedure*, not one chosen model — the final model must be refit on all data.

In [5]:
s = NestedCVSplitter(
    outer_splitter=get_splitter("k_fold", n_splits=3, shuffle=True, random_state=0),
    inner_splitter="random",
    random_state=0,
)
results = s.split_result(SMILES_POOL)
summarize(results, "NestedCVSplitter")

NestedCVSplitter: 3 SplitResult(s)
  [0] train=22 valid=5 test=14 discard=0 {}
  [1] train=22 valid=5 test=14 discard=0 {}
  [2] train=22 valid=6 test=13 discard=0 {}


<a id="5"></a>
## 5. 🌍 ExternalHoldoutSplitter — a truly independent test set

Forces a caller-supplied external dataset to be the entire test partition, appended after the original records (global indices `n .. n+m-1`) — useful for reporting performance on a set nobody could have accidentally tuned against.

| Parameter | Meaning |
|---|---|
| `X_external` | The external holdout set, in the same form as the main `X` |

> 💡 **Advantages:**
> - The one evaluation nobody can accidentally tune against, since the data was assembled independently.
> - Overlap checking is on by default, closing the step most often skipped when an "external" set quietly contains training compounds.
>
> ⚠️ **Pitfalls:**
> - "External" describes provenance, not chemistry — a set from the same vendor catalogue isn't external in any meaningful sense.
> - Index semantics change: results index into the *concatenated* array, not just the original `X`.
> - A single external set is one sample of one distribution — a good score is evidence, not proof.

In [6]:
external = ["CCCCCCCCCC", "c1ccc(I)cc1"]
s = ExternalHoldoutSplitter(X_external=external)
results = s.split_result(SMILES_POOL)
summarize(results, "ExternalHoldoutSplitter")
r = results[0]
print("n_records (pool + external) ->", r.n_records)

ExternalHoldoutSplitter: 1 SplitResult(s)
  [0] train=41 valid=0 test=2 discard=0 {}
n_records (pool + external) -> 43


<a id="6"></a>
## 6. 📏 ApplicabilityDomainSplitter — a distance-vs-performance curve

Produces a *series* of test "bands" at increasing nearest-neighbour distance from a shared train set — not one split, but a distance-vs-performance curve. It directly answers the question a deployed model actually needs answered: "reliable below distance *d*, degrading beyond it".

| Parameter | Meaning |
|---|---|
| `n_bands` | Number of equal-count bands the test set is cut into, ordered by increasing nearest-neighbour distance to train |
| `train_size` / `test_size` | Sizes for the initial (default `"random"`) train/test partition the bands are drawn from |

> 💡 **Advantages:**
> - Replaces the unanswerable "which split is correct?" with a measurement — error as a function of distance from the training set.
> - Works on top of any base split, so the curve applies equally to a random, scaffold, or time split.
>
> ⚠️ **Pitfalls:**
> - Bands are subsets of one test set, so each is small and noisy.
> - The distance statistic is metric- and fingerprint-dependent, so the curve's x-axis isn't comparable across representations.
> - A monotone-looking curve can be an artefact of a confound (e.g. molecular size increasing with distance).

In [7]:
s = ApplicabilityDomainSplitter(n_bands=3, train_size=0.6, test_size=0.4, random_state=0)
results = s.split_result(SMILES_POOL)
summarize(results, "ApplicabilityDomainSplitter")
print("all bands share the same train set:",
      len({tuple(r.train.tolist()) for r in results}) == 1)

ApplicabilityDomainSplitter: 3 SplitResult(s)
  [0] train=25 valid=0 test=6 discard=10 {'band_index': 0, 'distance_range': [0.0, 0.625]}
  [1] train=25 valid=0 test=5 discard=11 {'band_index': 1, 'distance_range': [0.625, 0.7272727489471436]}
  [2] train=25 valid=0 test=5 discard=11 {'band_index': 2, 'distance_range': [0.7777777910232544, 0.9090909361839294]}
all bands share the same train set: True


---
That's all 6 `protocol` splitters. They're deliberately orthogonal to *what* criterion decides train/valid/test — pick any splitter from another family as the `base_splitter`/`grouper`/`inner_splitter` argument and the protocol wraps around it unchanged.